In [73]:
from scipy.constants import electron_mass

mol_dir = "./mols"
basis_sets_dir = "./basis-sets"

particle_properties_file = "particle-properties.json"

In [74]:
e_basis_set = "6-31G"
n_basis_set = "DZSNB"

mol_name = "H2"

truncate_e = 0

mtx_elmt_threshold = 1e-7

In [75]:
import numpy as np
import json
import itertools

from scipy.sparse import coo_matrix, csr_matrix
from scipy.linalg import block_diag


from pyscf import gto, scf
from pyscf.lo import orth

from gbasis.wrappers import from_pyscf
from gbasis.parsers import parse_nwchem
from gbasis.parsers import make_contractions

from gbasis.integrals.overlap import overlap_integral
from gbasis.integrals.kinetic_energy import kinetic_energy_integral
from gbasis.integrals.electron_repulsion import electron_repulsion_integral

# Load properties of all possible particles (spin, fermion/boson, mass, charge, etc)
with open(particle_properties_file, "r") as file:
    particle_properties = json.load(file)

# Build molecule for PySCF
mol = gto.Mole()
mol.atom = mol_dir + '/' + mol_name + '.xyz'
mol.basis = e_basis_set
mol.build()

mol_zs = mol.atom_charges()
mol_symbs = [mol.atom_symbol(i) for i in range(mol.natm)] # Atomic symbols
mol_coords = mol.atom_coords()

# Run restricted Hartree Fock to get better orbitals (I might truncate the highest energy ones)
hf = scf.RHF(mol).run() # TODO: apparently there might be better choices of orbital to allow for truncations (FNO). look into?

# Load basis dictionary (atomic orbitals) for nuclear orbitals
n_basis_dict = parse_nwchem(basis_sets_dir + '/nuclear/' + n_basis_set + '.nw')

# Construct a dictionary of all the particle types that will be in our calculation, along with their orbitals and info like spin.
# Note: we will use the order of the dictionary. Python 3.7+ guarantees when we iterate, the dictionary will be ordered according to when the elements were added.
particles = {}

for i in range(mol.natm):
    symb = mol.atom_symbol(i)
    # If this is a particle type (nucleus) we haven't seen before
    if symb not in particles:
        # Add it to particle list
        particles[symb] = {}
        particles[symb]['coords'] = []
        particles[symb]['count'] = 0

    # Add its coordinates to the list
    particles[symb]['coords'].append(mol_coords[i])

    particles[symb]['count'] += 1

for symb in particles:
    # GBasis wants coords as numpy array
    particles[symb]['coords'] = np.array(particles[symb]['coords'])

    # Construct a basis w/ GBasis for each of the nuclear particles
    particles[symb]['basis'] = make_contractions(n_basis_dict, # Basis of (nuclear) AOs to use
                                                 [symb] * particles[symb]['count'], # Types of atoms (all the same)
                                                 particles[symb]['coords'], # Coordinates
                                                 coord_types='cartesian')

    # Transform to orthonormal orbitals
    overlap = overlap_integral(particles[symb]['basis'])
    particles[symb]['transform'] = orth.lowdin(overlap) # Symmetric orthonormalization of AOs

    # Number of spatial orbitals
    particles[symb]['no_spatial_orbitals'] = overlap.shape[0]

# Retrieve some important properties on each particle (spin, fermion/boson, mass, charge)
for symb in particles:
    particles[symb]['properties'] = particle_properties[symb]

# Add electrons to our particle list
particles['e'] = {}
particles['e']['basis'] = from_pyscf(mol) # Gbasis set of GTOs (gaussian type orbitals) for electronic particles
particles['e']['transform'] = hf.mo_coeff.T # transform to MOs that will be used for calculation
particles['e']['count'] = mol.nelectron  # Get number of electrons from PySCF, truncate off 10
particles['e']['no_spatial_orbitals'] = hf.mo_coeff.shape[0] - truncate_e

idx = 0

# Retrieve some important properties on each particle (spin, fermion/boson, mass, charge)
for symb in particles:
    particles[symb]['idx'] = idx
    idx += 1

    particles[symb]['properties'] = particle_properties[symb]

    # Number of spin orbitals, since we now have the particle's spin
    particles[symb]['no_spin_orbitals'] = particles[symb]['no_spatial_orbitals'] * particles[symb]['properties']['spin']

# Amount of particles we have
particle_types = len(particles)

# List of particle names for easy indexing
particle_names = [symb for symb in particles]

converged SCF energy = -1.12673396711657


In [76]:
# TODO: I should probably just do all the integrals at once by combining the bases of all particles.

# Our basis in particles[symb]['basis'] is a basis of spatial orbitals, so the integrals will be between spatial oritals.
# This class will take a matrix (or 4D array for 2-body interactions) of these spatial integrals and give us a matrix that indexes spin orbitals
# We use the convention defined below for indexing spin orbitals
#
# This class ASSUMES 2-body integrals are in CHEMIST'S NOTATION:
# data[i,j,k,l] = \int dr_1 dr_2 \chi_i^*(r_1) \chi_j(r_1) \hat{O}_2 \chi_k^*(r_2) \chi_l(r_2)
class IntegralSpinWrapper:
    # data: table of integrals (2D array for 1-body, 4D array for 2-body). Should be a numpy array, which it will be for integrals from GBasis
    # spin: number of spin states if 1-body, or 2-tuple of the number of spin states for each particle if 2-body
    def __init__(self, data, spin):
        self.data = data
        self.spin = spin

        self.is_two_body = len(data.shape) == 4

    def __getitem__(self, index):
        # If one-body interaction
        if not self.is_two_body:
            # Spin states must be equal.
            if index[0] % self.spin == index[1] % self.spin:
                return self.data[int(index[0] / self.spin), int(index[1] / self.spin)]

            # Otherwise by orthonormality the integral is 0
            else:
                return 0

        # If two-body interaction
        else:
            # Spin states must be equal for both coordinates:
            if (index[0] % self.spin[0] == index[1] % self.spin[0]) and (index[2] % self.spin[1] == index[3] % self.spin[1]):
                return self.data[int(index[0]/self.spin[0]), int(index[1]/self.spin[0]), int(index[2]/self.spin[1]), int(index[3]/self.spin[1])]

            # Otherwise by orthonormality the integral is 0
            else:
                return 0

In [77]:
# For full FCI, we will construct the Hamiltonian in the subspace of all states with only the correct particle numbers
# A basis for this space is constructed from N-particle determinants/permanents of the one-particle basis states for correct N

# Total number of states in this space
total_states = 1

# Number of permanents/determinants for each particle.
no_states = []

# For the Hamiltonian matrix, we will index the states as follows:
# Let N_i be the number of states for the i-th particle
# The state formed from the c_0 - th particle 1 permanent/determinant, c_1 - th particle 2 perminant/determinant, etc (c_i's zero indexed)
# Will get the index (c_0) + (c_1 * N_0) + (c_2 * N_1 * N_0) + (c_3 * N_2 * N_1 * N_0) + ...

# This works basically like a numeral base system where each digit has a different base (each digit is a particle).
# This array will contain the bases for each particle: [1, N_0, N_1 * N_0, ...]
bases = []

# Construct N-particle states
for symb in particles:
    particles[symb]['states'] = []

    # Construct all N-particle states for the correct N, in the forms of arrays of 1s and 0s. They will be indexed in the order we get them from itertools
    # States with the same spatial wave function will be grouped together. For example, for 4 spatial orbitals with A and B spin states, the significance of the bits will be
    # Bit number:       12345678
    # Spin:             ABABABAB
    # Spatial orbital:  11223344
    for indices in itertools.combinations(range(particles[symb]['no_spin_orbitals']), particles[symb]['count']):
        array = [0] * particles[symb]['no_spin_orbitals']
        for index in indices:
            array[index] = 1

        particles[symb]['states'].append(array)

    particles[symb]['no_states'] = len(particles[symb]['states'])
    particles[symb]['base'] = total_states

    no_states.append(particles[symb]['no_states'])
    bases.append(particles[symb]['base'])
    total_states *= particles[symb]['no_states']

bases.append(total_states) # Having this extra element will be helpful in construct_1_particle_interaction


In [78]:
# Construct the contribution to the FCI Hamiltonian from a one-body interaction (e.g. <e_1 | KE | e_3> where e_i are electronic basis states)
# We can have any state for the remaining particles so we will iterate over all possible determinants for every other particle

# particle_no: index in the dictionary of the particle (use particle_names for indexing)
# bra: index of the bra state in particle particle_no (this is a determinant/permanent, not a single particle state!)
# ket: inex of the ket state                          (this is a determinant/permanent, not a single particle state!)
# value: value of <bra | O_1 | ket>
def construct_1_particle_interaction(particle, bra, ket, value):
    if abs(value) < mtx_elmt_threshold:
        return csr_matrix((total_states, total_states))

    # The idea here is that the states (in our huge big space) that involve this specific matrix element <bra | O_1 | ket> are those that look like
    # bra: |  ANY  |  ANY  |  ...  |  bra  |  ... |  ANY  |
    # ket: |  ANY^ |  ANY^ |  ...  |  ket  |  ... |  ANY  |
    #        ptcl 1  ptcl 2   ..   ptcl ptcl_no ..    ptcl particle_types
    # where the boxes are our choice of determinant/permanent for each particle type, and between the bra and the ket, have to be the same for all
    # particle types other than the one labeled by ptcl_no (the argument, particle_no).
    # So we will iterate over all possible choices of determinants for the other particles

    # Lists of the row and column indices and the values (will all be the same, value) in the Hamiltonian matrix.
    mtx_rows = []
    mtx_cols = []
    mtx_values = []

    base = particle['base']
    no_states = particle['no_states']

    # Iterate all over choices of determinant/permanent for the particles coming BEFORE ptcl_no
    for i in range(base):
        # Iterate all over choices of determinant/permanent for the particles coming AFTER ptcl_no
        for j in range(int(total_states/(base*no_states))):
            # Construct the indices in the Hamiltonian matrix (base convention) for these bra and ket states
            bra_idx = i + (bra * base) + (j * base * no_states)
            ket_idx = i + (ket * base) + (j * base * no_states)

            # Add to the list of elements
            mtx_rows.append(bra_idx)
            mtx_cols.append(ket_idx)
            mtx_values.append(value)

    return coo_matrix((mtx_values, (mtx_rows, mtx_cols)), shape=(total_states, total_states))#.tocsr()

# Construct the contribution to the FCI Hamiltonian from a two-body interaction between two DISTINCT PARTICLES(e.g. <e_1n_2 | V | e_3e_4> where e_i are one-particle electronic basis states and n_i are one-particle basis states for some other particle)
# We can have any state for the remaining particles so we will iterate over all possible determinants for every other particle
# Note that two-body interactions between particles of the same type should be handled by construct_1_particle
# NOTE: particle1_no MUST be less than particle2_no for this to work.

# particle1_no: index in the dictionary for particle type 1 (use particle_names for indexing)
# particle2_no: index in the dictionary for particle type 2
# bra1: index of the bra state in particle particle1_no (this is a determinant/permanent, not a single particle state!)
# ket1: inex of the ket state                           (this is a determinant/permanent, not a single particle state!)
# etc...
# value: value of <bra | O_1 | ket>
def construct_2_particle_interaction(particle1, particle2, bra1, ket1, bra2, ket2, value):
    if abs(value) < mtx_elmt_threshold:
        return csr_matrix((total_states, total_states))

    # The idea here is that the states (in our huge big space) that involve this specific matrix element <bra1 | O_1 | ket> are those that look like
    # bra: |  ANY  |  ANY  |  ...  |  bra1  |  ... |  bra2  |  ... |  ANY  |
    # ket: |  ANY^ |  ANY^ |  ...  |  ket1  |  ... |  bra2  |  ... |  ANY  |
    #        ptcl 1  ptcl 2   ..   ptcl ptc1l_no ..    ptcl2_no particle_types
    # where the boxes are our choice of determinant/permanent for each particle type, and between the bra and the ket, have to be the same for all
    # particle types other than the one labeled by ptcl_no (the argument, particle_no).
    # So we will iterate over all possible choices of determinants for the other particles

    # Lists of the row and column indices and the values (will all be the same, value) in the Hamiltonian matrix.
    mtx_rows = []
    mtx_cols = []
    mtx_values = []

    base1 = particle1['base']
    base2 = particle2['base']

    no_states1 = particle1['no_states']
    no_states2 = particle2['no_states']

    # Iterate all over choices of determinant/permanent for the particles coming BEFORE ptcl1_no
    for i in range(base1):
        # Iterate all over choices of determinant/permanent for the particles coming AFTER ptcl1_no but BEFORE ptcl2_no
        for j in range(int(base2/(base1*no_states1))):
            # Iterate all over choices of determinant/permanent for the particles coming AFTER ptcl2_no
            for k in range(int(total_states/(base2*no_states2))):
                # Construct the indices in the Hamiltonian matrix (base convention) for these bra and ket states
                bra_idx = i + (bra1 * base1) + (j * base1 * no_states1) + (bra2 * base2) + (k * base2 * no_states2)
                ket_idx = i + (ket1 * base1) + (j * base1 * no_states1) + (ket2 * base2) + (k * base2 * no_states2)

                # Add to the list of elements
                mtx_rows.append(bra_idx)
                mtx_cols.append(ket_idx)
                mtx_values.append(value)

    return coo_matrix((mtx_values, (mtx_rows, mtx_cols)), shape=(total_states, total_states)).tocsr()

In [79]:
# Given a certain N-particle state index, construct an array of the indices of all states differing by d one-particle states, combined with info on which states are different and what the parity factor is after aligning up all the common states
def get_diff_states(particle, state_idx, d):
    state = particle['states'][state_idx]

    # Indices of the occupied & unoccupied one-particle states for this N-particle state
    occupied = [i for i in range(len(state)) if state[i] == 1]
    unoccupied = [i for i in range(len(state)) if state[i] == 0]

    new_states = []

    deoccupy_comb = itertools.combinations(occupied, d)
    occupy_comb = itertools.combinations(unoccupied, d)

    comb = itertools.product(itertools.combinations(occupied, d), # Which d one-particle states to de-occupy in the new state
                             itertools.combinations(unoccupied, d)) # Which d one-particle states to occupy in the new state

    for deoccupy, occupy in comb:
        # Construct new state
        new_state = state.copy()
        for d in deoccupy:
            new_state[d] = 0 # Deoccupy these states

        for o in occupy:
            new_state[o] = 1 # Occupy these states

        # States in common
        common = [i for i in occupied if i not in deoccupy]

        # Get index of this new state
        new_state_idx = particle['states'].index(new_state)

        # How many permutations to align the old state and new state's determinants?
        perms = sum([sum(min(d, o) < c < max(d, o) for c in common) for d, o in zip(deoccupy, occupy)])

        # Fermion exchange parity from aligning the determinants
        parity = 1 if perms % 2 == 0 else -1

        # Package all this info together
        new_states.append((new_state_idx, common, deoccupy, occupy, parity))

    return new_states

In [80]:
print(total_states)

784


In [81]:
# We will construct the Hamiltonian matrix in this basis, one interaction at a time

t_mtx = csr_matrix((total_states,total_states)) # Create empty sparse matrix of the appropriate size for the KE operator

t1s = []

# One-body "interactions" (kinetic energy)
for symb in particles:
    particle = particles[symb]

    mass = particle['properties']['mass'] # Mass of the particle
    spin = particle['properties']['spin'] # Spin of the particle

    no_states = particle['no_states'] # Number of N-particle states

    # Get kinetic energy integrals
    ke_int = kinetic_energy_integral(particle['basis'], particle['transform']) / mass
    particle['ke_int'] = ke_int # Store them

    # Get the spin orbital integrals
    ke_int_spin = IntegralSpinWrapper(ke_int, spin)

    t1 = (particle, [])

    # For all N-particle bras, use Slater-Condon rules to calculate matrix elements
    for bra_idx in range(no_states):
        # Matrix elements for states that differ by 0 one-particle states
        for ket_idx, common, deoccupy, occupy, parity in get_diff_states(particle, bra_idx, 0):
            elmt = sum([ke_int_spin[c, c] for c in common])
            elmt *= parity

            #t_mtx += construct_1_particle_interaction(particle, bra_idx, ket_idx, elmt)
            if abs(elmt) > mtx_elmt_threshold:
                t1[1].append((bra_idx, ket_idx, elmt))

        # Matrix elements for states that differ by 1 one-particle state
        for ket_idx, common, deoccupy, occupy, parity in get_diff_states(particle, bra_idx, 1):
            elmt = ke_int_spin[deoccupy[0], occupy[0]]
            elmt *= parity

            #t_mtx += construct_1_particle_interaction(particle, bra_idx, ket_idx, elmt)
            if abs(elmt) > mtx_elmt_threshold:
                t1[1].append((bra_idx, ket_idx, elmt))

    t1s.append(t1)

In [82]:
print(particles['e']['no_spin_orbitals'])

8


In [83]:

"""
    for bra_idx in range(particle['no_states']):
        bra = particle['states'][bra_idx]

        # Indices of the occupied & unoccupied one-particle states for this bra
        occupied = [i for i in range(len(bra)) if bra[i] == 1]
        unoccupied = [i for i in range(len(bra)) if bra[i] == 0]

        print(occupied, unoccupied)

        # First, the element <XXXbraYYY|T|XXXbraYYY> (we will consider all choices of states for the other particles, XXX YYY, by calling construct_1_particle_interaction).
        # That is, the bra and the ket are the same state
        # According to Szabo, this is sum_i [i | O_1 | i] for all occupied orbitals i.
        bra_bra_elmt = 0

        for i in occupied: # Iterate through all occupied one-particle states
            bra_bra_elmt += ke_int_spin[i, i] # Add [i | KE | i]

        t_mtx += construct_1_particle_interaction(particle_no, bra_idx, bra_idx, bra_bra_elmt)

        # Now, we consider elements of the form <XXXbraYYY|T|XXXketYYY> for all ket. Ket must differ from bra by at most one one-particle state,
        # So we will iterate over all states that differ by only one one-particle state.
        # Szabo says this matrix element will be [i | KE | j], where i is the one one-particle state that's only in the bra, and j is the state that's only in the ket
        for bra_1p in occupied: # Iterate through all occupied one-particle states. This is the state we will not include in the ket
            for ket_1p in unoccupied: # This is the state we will replace bra_1p with in our ket state
                # Construct ket state
                ket = bra.copy()
                ket[bra_1p] = 0
                ket[ket_1p] = 1

                #print(ket, bra_1p, ket_1p)

                ket_idx = particle['states'].index(ket)

                bra_ket_elmt = ke_int_spin[bra_1p, ket_1p]
                #print(bra_ket_elmt)

                t_mtx += construct_1_particle_interaction(particle_no, bra_idx, ket_idx, bra_ket_elmt)


        #print(bra)
        #print(h_mtx)

        if(bra_idx > 0):
            break
"""
v_mtx = coo_matrix((total_states,total_states))
v1s = []

# Two-body interactions between particles of the SAME type. TODO: implement boson interactions with their different exchange behavior
for symb in particles:
    particle = particles[symb]

    spin = particle['properties']['spin'] # Spin of the particle
    no_states = particle['no_states'] # Number of N-particle states

    # Get two body integrals (coulomb force)
    cmb_int = electron_repulsion_integral(particle['basis'], particle['transform'], notation='chemist') * (particle['properties']['charge'] ** 2)
    particle['cmb_int'] = cmb_int # Store them

    # Get the spin orbital integrals
    cmb_int_spin = IntegralSpinWrapper(cmb_int, (spin, spin))

    v1 = (particle, [])

    # For all N-particle bras, use Slater-Condon rules to calculate matrix elements
    for bra_idx in range(no_states):
        print(bra_idx)
        # Matrix elements for states that differ by 0 one-particle states
        for ket_idx, common, deoccupy, occupy, parity in get_diff_states(particle, bra_idx, 0):
            elmt = sum([cmb_int_spin[i,i,j,j]-cmb_int_spin[i,j,j,i] for i, j in itertools.combinations(common, 2)])
            elmt *= parity

            #v_mtx += construct_1_particle_interaction(particle, bra_idx, ket_idx, elmt)
            if abs(elmt) > mtx_elmt_threshold:
                v1[1].append((bra_idx, ket_idx, elmt))

        # Matrix elements for states that differ by 1 one-particle state
        for ket_idx, common, deoccupy, occupy, parity in get_diff_states(particle, bra_idx, 1):
            elmt = sum([cmb_int_spin[deoccupy[0], occupy[0], i, i] - cmb_int_spin[deoccupy[0], i, i, occupy[0]] for i in common])
            elmt *= parity

            #v_mtx += construct_1_particle_interaction(particle, bra_idx, ket_idx, elmt)
            if abs(elmt) > mtx_elmt_threshold:
                v1[1].append((bra_idx, ket_idx, elmt))

        # Matrix elements for states that differ by 2 one-particle state
        for ket_idx, common, deoccupy, occupy, parity in get_diff_states(particle, bra_idx, 2):
            elmt = cmb_int_spin[deoccupy[0], occupy[0], deoccupy[1], occupy[1]] - cmb_int_spin[deoccupy[0], occupy[1], deoccupy[1], occupy[0]]
            elmt *= parity

            #v_mtx += construct_1_particle_interaction(particle, bra_idx, ket_idx, elmt)
            if abs(elmt) > mtx_elmt_threshold:
                v1[1].append((bra_idx, ket_idx, elmt))

    v1s.append(v1)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27


In [84]:
v_mtx2 = v_mtx.copy()

In [85]:
v2s = []

# Two-body interactions between particles of the SAME type. TODO: implement boson interactions with their different exchange behavior
for symb1, symb2 in itertools.combinations(particles, 2):
    particle1 = particles[symb1]
    particle2 = particles[symb2]

    no_states1 = particle1['no_states']
    no_states2 = particle2['no_states']

    b1 = particle1['no_spatial_orbitals'] # number of spatial basis functions for each particle
    b2 = particle2['no_spatial_orbitals']

    # Get two body integrals (coulomb force). Have to combine the bases and then select only the integrals between the two particles
    cmb_int = electron_repulsion_integral(particle1['basis'] + particle2['basis'], transform=block_diag(particle1['transform'], particle2['transform']), notation='chemist')[0:b1, 0:b1, b1:b1+b2, b1:b1+b2]
    cmb_int *= particle1['properties']['charge'] * particle2['properties']['charge']
        #particle['cmb_int'] = cmb_int # Store them

    # Get the spin orbital integrals
    cmb_int_spin = IntegralSpinWrapper(cmb_int, (particle1['properties']['spin'], particle2['properties']['spin']))

    v2 = ((particle1, particle2), [])

    # For all N-particle bras, use Slater-Condon rules to calculate matrix elements
    for bra1_idx in range(no_states1):
        for bra2_idx in range(no_states2):
            print(bra1_idx, bra2_idx)
            # Matrix elements for states that differ by 0 one-particle states for both bras
            for ket1_idx, common1, deoccupy1, occupy1, parity1 in get_diff_states(particle1, bra1_idx, 0):
                for ket2_idx, common2, deoccupy2, occupy2, parity2 in get_diff_states(particle2, bra2_idx, 0):
                    elmt = sum([cmb_int_spin[i,i,j,j] for i, j in itertools.product(common1, common2)])
                    elmt *= parity1 * parity2

                    #v_mtx += construct_2_particle_interaction(particle1, particle2, bra1_idx, ket1_idx, bra2_idx, ket2_idx, elmt)
                    if abs(elmt) > mtx_elmt_threshold:
                        v2[1].append((bra1_idx, ket1_idx, bra2_idx, ket2_idx, elmt))

            # States that differ by 0 for bra1, 1 for bra2
            for ket1_idx, common1, deoccupy1, occupy1, parity1 in get_diff_states(particle1, bra1_idx, 0):
                for ket2_idx, common2, deoccupy2, occupy2, parity2 in get_diff_states(particle2, bra2_idx, 1):
                    elmt = sum([cmb_int_spin[i,i,deoccupy2[0],occupy2[0]] for i in common1])
                    elmt *= parity1 * parity2

                    #v_mtx += construct_2_particle_interaction(particle1, particle2, bra1_idx, ket1_idx, bra2_idx, ket2_idx, elmt)
                    if abs(elmt) > mtx_elmt_threshold:
                        v2[1].append((bra1_idx, ket1_idx, bra2_idx, ket2_idx, elmt))

            # States that differ by 1 for bra1, 0 for bra2
            for ket1_idx, common1, deoccupy1, occupy1, parity1 in get_diff_states(particle1, bra1_idx, 1):
                for ket2_idx, common2, deoccupy2, occupy2, parity2 in get_diff_states(particle2, bra2_idx, 0):
                    elmt = sum([cmb_int_spin[deoccupy1[0],occupy1[0],j,j] for j in common2])
                    elmt *= parity1 * parity2

                    #v_mtx += construct_2_particle_interaction(particle1, particle2, bra1_idx, ket1_idx, bra2_idx, ket2_idx, elmt)
                    if abs(elmt) > mtx_elmt_threshold:
                        v2[1].append((bra1_idx, ket1_idx, bra2_idx, ket2_idx, elmt))

            # States that differ by 1 for bra1 and bra2
            for ket1_idx, common1, deoccupy1, occupy1, parity1 in get_diff_states(particle1, bra1_idx, 1):
                for ket2_idx, common2, deoccupy2, occupy2, parity2 in get_diff_states(particle2, bra2_idx, 1):
                    elmt = cmb_int_spin[deoccupy1[0],occupy1[0],deoccupy2[0],occupy2[0]]
                    elmt *= parity1 * parity2

                    #v_mtx += construct_2_particle_interaction(particle1, particle2, bra1_idx, ket1_idx, bra2_idx, ket2_idx, elmt)
                    if abs(elmt) > mtx_elmt_threshold:
                        v2[1].append((bra1_idx, ket1_idx, bra2_idx, ket2_idx, elmt))



    v2s.append(v2)

0 0
0 1
0 2
0 3
0 4
0 5
0 6
0 7
0 8
0 9
0 10
0 11
0 12
0 13
0 14
0 15
0 16
0 17
0 18
0 19
0 20
0 21
0 22
0 23
0 24
0 25
0 26
0 27
1 0
1 1
1 2
1 3
1 4
1 5
1 6
1 7
1 8
1 9
1 10
1 11
1 12
1 13
1 14
1 15
1 16
1 17
1 18
1 19
1 20
1 21
1 22
1 23
1 24
1 25
1 26
1 27
2 0
2 1
2 2
2 3
2 4
2 5
2 6
2 7
2 8
2 9
2 10
2 11
2 12
2 13
2 14
2 15
2 16
2 17
2 18
2 19
2 20
2 21
2 22
2 23
2 24
2 25
2 26
2 27
3 0
3 1
3 2
3 3
3 4
3 5
3 6
3 7
3 8
3 9
3 10
3 11
3 12
3 13
3 14
3 15
3 16
3 17
3 18
3 19
3 20
3 21
3 22
3 23
3 24
3 25
3 26
3 27
4 0
4 1
4 2
4 3
4 4
4 5
4 6
4 7
4 8
4 9
4 10
4 11
4 12
4 13
4 14
4 15
4 16
4 17
4 18
4 19
4 20
4 21
4 22
4 23
4 24
4 25
4 26
4 27
5 0
5 1
5 2
5 3
5 4
5 5
5 6
5 7
5 8
5 9
5 10
5 11
5 12
5 13
5 14
5 15
5 16
5 17
5 18
5 19
5 20
5 21
5 22
5 23
5 24
5 25
5 26
5 27
6 0
6 1
6 2
6 3
6 4
6 5
6 6
6 7
6 8
6 9
6 10
6 11
6 12
6 13
6 14
6 15
6 16
6 17
6 18
6 19
6 20
6 21
6 22
6 23
6 24
6 25
6 26
6 27
7 0
7 1
7 2
7 3
7 4
7 5
7 6
7 7
7 8
7 9
7 10
7 11
7 12
7 13
7 14
7 15
7 16
7 17
7 18
7 19


In [86]:
print(len(v2s[2][1]))

IndexError: list index out of range

In [47]:
import pickle

with open("v2s-2.pkl", "wb") as file:
    pickle.dump(v2s[2][1], file)

In [36]:
h_mtx = t_mtx + v_mtx

In [87]:
no_statess=tuple([particles[symb]['no_states'] for symb in particles])

def matvec(v):
    print('Called!')
    vn = v.reshape(no_statess, order='F')
    vr = np.zeros(no_statess)

    for particle, elmts in t1s:
        slice_bra = [slice(None)] * particle_types
        slice_ket = [slice(None)] * particle_types

        for bra_idx, ket_idx, elmt in elmts:
            slice_bra[particle['idx']] = bra_idx
            slice_ket[particle['idx']] = ket_idx

            vr[tuple(slice_ket)] += vn[tuple(slice_bra)] * elmt

    for particle, elmts in v1s:
        slice_bra = [slice(None)] * particle_types
        slice_ket = [slice(None)] * particle_types

        for bra_idx, ket_idx, elmt in elmts:
            slice_bra[particle['idx']] = bra_idx
            slice_ket[particle['idx']] = ket_idx

            vr[tuple(slice_ket)] += vn[tuple(slice_bra)] * elmt

    for particles, elmts in v2s:
        particle1 = particles[0]
        particle2 = particles[1]

        slice_bra = [slice(None)] * particle_types
        slice_ket = [slice(None)] * particle_types

        for bra1_idx, ket1_idx, bra2_idx, ket2_idx, elmt in elmts:
            #print(bra_idx, ket_idx)
            slice_bra[particle1['idx']] = bra1_idx
            slice_ket[particle1['idx']] = ket1_idx
            slice_bra[particle2['idx']] = bra2_idx
            slice_ket[particle2['idx']] = ket2_idx
            #print(slice_bra, slice_ket)
            #print(vn)

            vr[tuple(slice_ket)] += vn[tuple(slice_bra)] * elmt
    return vr.reshape(total_states, order='F')

In [59]:
v = np.zeros(total_states, order='F')
v[0] = 1

print(matvec(v))

[-6.7629673   0.         -0.02353634 ...  0.          0.
  0.        ]


In [88]:
import scipy as sp
#h_mtx = t_mtx + v_mtx
#h_eigvals, h_eigvecs = sp.sparse.linalg.eigsh(h_mtx, k=20, which='SA')

from scipy.sparse.linalg import LinearOperator
A = LinearOperator(shape=(total_states, total_states), matvec=matvec, dtype=float)

a_eigvals, a_eigvecs = sp.sparse.linalg.eigsh(A, k=25, which='SA', tol=1e-6, maxiter=250)

Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!
Called!


In [89]:
a_eigvals

array([-1.06612452, -1.06612452, -1.06612452, -1.06612452, -1.01532449,
       -1.01532449, -1.01532449, -1.01532449, -1.01497468, -1.01497468,
       -1.01497468, -1.01497468, -0.96380848, -0.96380848, -0.96380848,
       -0.96380848, -0.68014356, -0.68014356, -0.68014356, -0.68014356,
       -0.63363118, -0.63363118, -0.63363118, -0.63311854, -0.63311854])

In [55]:
h2_4 = [-1.0964299  -1.0964299  -1.0964299  -1.0964299  -1.07135739 -1.07135739
 -1.07135739 -1.07135739 -1.07077338 -1.07077338 -1.07077338 -1.07077338
 -1.06827109 -1.06827109 -1.06827109 -1.06827109 -1.05622724 -1.05622724
 -1.05622724 -1.05474387 -1.05474387 -1.05474387 -1.05257179 -1.05257179
 -1.05257179] # h2 eigenvalues for DZSPN + (def2_SVP - 4)

h2_0 = [-1.11013085 -1.11013085 -1.11013085 -1.11013085 -1.08094146 -1.08094146
 -1.08090026 -1.08090026 -1.07947315 -1.07947315 -1.07152197 -1.07152197
 -1.06768781 -1.06768781 -1.06760151 -1.06760151 -1.05004097 -1.04984186
 -1.04968478 -1.04542128 -1.04513957 -1.03985392 -1.03675355 -1.03626153
 -1.03612331] # h2 eigenvalues for DZSPN + (def2_SVP - 4)

h2_blah = ([-1.06612452, -1.06612452, -1.06612452, -1.06612452, -1.01532449,
       -1.01532449, -1.01532449, -1.01532449, -1.01497468, -1.01497468,
       -1.01497468, -1.01497468, -0.96380848, -0.96380848, -0.96380848,
       -0.96380848, -0.68014356, -0.68014356, -0.68014356, -0.68014356,
       -0.63363118, -0.63363118, -0.63363118, -0.63311854, -0.63311854]) # DZSNB +6-31G

lih_dzsnb_3_21g_trunc4 = ([-6.97503307, -6.97503307, -6.97503307, -6.97503307, -6.92800156,
       -6.92800156, -6.92800156, -6.92800156, -6.86632007, -6.86632007,
       -6.86632007, -6.85831466, -6.85831466, -6.85831466, -6.82734175,
       -6.82734175, -6.82413721, -6.82413721, -6.81870311, -6.81870311,
       -6.81595908, -6.81595908, -6.78521796, -6.78521796, -6.77908297])

In [40]:
-1.08094146+1.11013085

0.029189389999999982

In [114]:
d1s = []
p_ord = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]])
# One-body "interactions" (kinetic energy)
for symb in particles:
    particle = particles[symb]

    mass = particle['properties']['mass'] # Mass of the particle
    spin = particle['properties']['spin'] # Spin of the particle

    no_states = particle['no_states'] # Number of N-particle states

    # Get kinetic energy integrals
    d_int = moment_integral(particle['basis'], particle['transform']) / mass

    # Get the spin orbital integrals
    d_int_spin = IntegralSpinWrapper(d_int, spin)

    d1 = (particle, [])

    # For all N-particle bras, use Slater-Condon rules to calculate matrix elements
    for bra_idx in range(no_states):
        # Matrix elements for states that differ by 0 one-particle states
        for ket_idx, common, deoccupy, occupy, parity in get_diff_states(particle, bra_idx, 0):
            elmt = sum([ke_int_spin[c, c] for c in common])
            elmt *= parity

            #t_mtx += construct_1_particle_interaction(particle, bra_idx, ket_idx, elmt)
            if abs(elmt) > mtx_elmt_threshold:
                t1[1].append((bra_idx, ket_idx, elmt))

        # Matrix elements for states that differ by 1 one-particle state
        for ket_idx, common, deoccupy, occupy, parity in get_diff_states(particle, bra_idx, 1):
            elmt = ke_int_spin[deoccupy[0], occupy[0]]
            elmt *= parity

            #t_mtx += construct_1_particle_interaction(particle, bra_idx, ket_idx, elmt)
            if abs(elmt) > mtx_elmt_threshold:
                t1[1].append((bra_idx, ket_idx, elmt))

    d1s.append(t1)

In [49]:

"""
# Two-body interactions between particles of the SAME type. TODO: implement boson interactions with their different exchange behavior
for particle_no in range(particle_types):
    print("--------------", particle_no, "----------------")
    symb = particle_names[particle_no]
    particle = particles[symb]

    spin = particle['properties']['spin'] # Spin of the particle

    # Get two body integrals (coulomb force)
    cmb_int = electron_repulsion_integral(particle['basis'], particle['transform'], notation='chemist') * (particle['properties']['charge'] ** 2)
    particle['cmb_int'] = cmb_int # Store them

    # Get the spin orbital integrals
    cmb_int_spin = IntegralSpinWrapper(cmb_int, (spin, spin))

    # For all N-particle bras, use Slater-Condon rules to calculate matrix elements
    for bra_idx in range(particle['no_states']):

        bra = particle['states'][bra_idx]

        # Indices of the occupied & unoccupied one-particle states for this bra
        occupied = [i for i in range(len(bra)) if bra[i] == 1]
        unoccupied = [i for i in range(len(bra)) if bra[i] == 0]

        print("~~~~~~~", occupied)

        # First, the element <XXXbraYYY|V|XXXbraYYY> (we will consider all choices of states for the other particles, XXX YYY, by callling construct_1_particle_interaction.
        # That is, the bra and the kets are the same state
        # According to Szabo, this is sum_i sum_{j<i} [ii|jj] - [ij|ji] (chemists notation)
        bra_bra_elmt = 0

        for i in range(len(occupied)):
            for j in range(i):
                i_state = occupied[i]
                j_state = occupied[j]

                print(i_state, j_state)

                bra_bra_elmt += cmb_int_spin[i,i,j,j] - cmb_int_spin[i,j,j,i]

        v_mtx += construct_1_particle_interaction(particle_no, bra_idx, bra_idx, bra_bra_elmt)

        # Now for states where the bra differs by only one one-particle state:
        for bra_1p in occupied:
            for ket_1p in unoccupied:
                # Construct ket state
                ket = bra.copy()
                ket[bra_1p] = 0
                ket[ket_1p] = 1

                # Get its index
                ket_idx = particle['states'].index(ket)

                # We require the determinants to have the same ordering of states except for the one state that's different. Since this won't be the case, we need to include a parity factor
                # Number of permutations required to align the states = number of states in COMMON between the two different states
                perms = sum(min(bra_1p, ket_1p) < o < max(bra_1p, ket_1p) for o in occupied)

                # Parity obtained after aligning the states up
                parity = 1 if perms % 2 == 0 else -1

                bra_ket_elmt = 0

                for o in occupied:
                    # Only iterate over states that are in common between the two
                    if o != bra_1p:
                        bra_ket_elmt += cmb_int_spin[bra_1p, ket_1p, o, o] - cmb_int_spin[bra_1p, o, o, ket_1p]

                bra_ket_elmt *= parity

                v_mtx += construct_1_particle_interaction(particle_no, bra_idx, ket_idx, bra_ket_elmt)

        # Now for states where the bra differs from the ket by two one-particle states:
        for bra1_1p, bra2_1p in itertools.combinations(occupied, 2):
            for ket1_1p, ket2_1p in itertools.combinations(unoccupied, 2):
                print(bra1_1p, bra2_1p, '----', ket1_1p, ket2_1p)

                perms = sum(min(bra1_1p, ket1_1p) < o < max(bra1_1p, ket1_1p) for o in occupied) + sum(min(bra2_1p, ket2_1p) < o < max(bra2_1p, ket2_1p) for o in occupied)

                parity = 1 if perms % 2 == 0 else -1

                # Construct ket
                ket = bra.copy()
                ket[bra1_1p] = 0
                ket[bra2_1p] = 0
                ket[ket1_1p] = 1
                ket[ket2_1p] = 1

                ket_idx = particle['states'].index(ket)

                bra_ket_elmt = parity * (cmb_int_spin[bra1_1p, ket1_1p, bra2_1p, ket2_1p] - cmb_int_spin[bra1_1p, ket2_1p, bra2_1p, ket1_1p])

                v_mtx += construct_1_particle_interaction(particle_no, bra_idx, ket_idx, bra_ket_elmt)


        if(bra_idx > 0):
            break



# Two-body interactions between particles of DIFFERENT type.
for particle2_no in range(particle_types):
    for particle1_no in range(particle2_no): # so that particle1_no < particle2_no
        symb1 = particle_names[particle1_no]
        symb2 = particle_names[particle2_no]
        print("--------------", particle1_no, symb1, ',', particle2_no, symb2, "----------------")
        particle1 = particles[symb1]
        particle2 = particles[symb2]

        spin1 = particle1['properties']['spin'] # Spin of the particle
        spin2 = particle2['properties']['spin'] #

        b1 = particle1['no_spatial_orbitals'] # number of spatial basis functions for each particle
        b2 = particle2['no_spatial_orbitals']

        # Get two body integrals (coulomb force). Have to combine the bases and then select only the integrals between the two particles
        cmb_int = electron_repulsion_integral(particle1['basis'] + particle2['basis'], transform=block_diag(particle1['transform'], particle2['transform']), notation='chemist')[0:b1, 0:b1, b1:b1+b2, b1:b1+b2]
        cmb_int *= particle1['properties']['charge'] * particle2['properties']['charge']
        #particle['cmb_int'] = cmb_int # Store them

        # Get the spin orbital integrals
        cmb_int_spin = IntegralSpinWrapper(cmb_int, (spin1, spin2))

        # Iterate over all bra N-particle states for these 2 particles
        for bra1_idx in range(particle1['no_states']):
            bra1 = particle1['states'][bra1_idx]

            # Indices of the occupied & unoccupied one-particle states for this bra
            occupied1 = [i for i in range(len(bra1)) if bra1[i] == 1]
            unoccupied1 = [i for i in range(len(bra1)) if bra1[i] == 0]

            for bra2_idx in range(particle2['no_states']):
                bra2 = particle2['states'][bra2_idx]

                # Indices of the occupied & unoccupied one-particle states for this bra
                occupied2 = [i for i in range(len(bra2)) if bra2[i] == 1]
                unoccupied2 = [i for i in range(len(bra2)) if bra2[i] == 0]

            bra_bra_elmt = 0

            # Rule if all states are the same between bra and ket: sum_i,j [ii|jj] for all occupied states i, j
            for o1 in occupied1:
                for o2 in occupied2:
                    bra_bra_elmt += cmb_int_spin[o1, o1, o2, o2]

            v_mtx += construct_2_particle_interaction(particle1_no, particle2_no, o1, o1, o2, o2, bra_bra_elmt)
"""

-------------- 0 H , 1 Li ----------------
-------------- 0 H , 2 e ----------------


KeyboardInterrupt: 

In [36]:
cmb_int = particles['e']['cmb_int']

print(v_mtx[0,0])
print(cmb_int[0,0,0,0] + cmb_int[1,1,1,1] + 4*cmb_int[0,0,1,1] - 2*
    cmb_int[0,1,1,0])

#print(v_mtx[0,512])
#print(base_breakdown(1024))
#print(particles['e']['states'][2])


TypeError: 'coo_matrix' object is not subscriptable

In [33]:
8#print(h_mtx[2,2])
H_ke = kinetic_energy_integral(particles['H']['basis'], particles['H']['transform'])
Li_ke = kinetic_energy_integral(particles['Li']['basis'], particles['Li']['transform'])
e_ke = kinetic_energy_integral(particles['e']['basis'], particles['e']['transform'])

H_mass = particles['H']['properties']['mass']
Li_mass = particles['Li']['properties']['mass']
e_mass = particles['e']['properties']['mass']

c = H_ke[0,0] / H_mass + Li_ke[0,0] / Li_mass + 2*(e_ke[0,0] + e_ke[1,1]) / e_mass
d = H_ke[0,1] / H_mass
e = H_ke[1,1] / H_mass + Li_ke[0,0] / Li_mass + 2*(e_ke[0,0] + e_ke[1,1]) / e_mass


print(c)
print(t_mtx[0,0])
print(d)
print(t_mtx[0,2])
print(e)
print(t_mtx[2,2])

8.038786322819048
8.038786322819048
-0.013239125947411641
-0.013239125947411641
8.00564820193124
8.00564820193124


In [37]:
print(bases)

[1, 16, 512, 253440]


In [20]:
print()

2


In [27]:
print(particles.keys())

dict_keys(['H', 'Li', 'e'])


In [31]:
print(particles['e']['states'][0])

[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [13]:
print(len(ke_int.shape))

2


In [21]:
prendus = construct_2_particle_interaction(particles['H'],particles['e'],3,1,0,4,5).tocoo()
[(base_breakdown(row), base_breakdown(col)) for row, col in list(zip(prendus.row, prendus.col))]

[([3, 0, 0], [1, 0, 4]),
 ([3, 1, 0], [1, 1, 4]),
 ([3, 2, 0], [1, 2, 4]),
 ([3, 3, 0], [1, 3, 4]),
 ([3, 4, 0], [1, 4, 4]),
 ([3, 5, 0], [1, 5, 4]),
 ([3, 6, 0], [1, 6, 4]),
 ([3, 7, 0], [1, 7, 4]),
 ([3, 8, 0], [1, 8, 4]),
 ([3, 9, 0], [1, 9, 4]),
 ([3, 10, 0], [1, 10, 4]),
 ([3, 11, 0], [1, 11, 4]),
 ([3, 12, 0], [1, 12, 4]),
 ([3, 13, 0], [1, 13, 4]),
 ([3, 14, 0], [1, 14, 4]),
 ([3, 15, 0], [1, 15, 4]),
 ([3, 16, 0], [1, 16, 4]),
 ([3, 17, 0], [1, 17, 4]),
 ([3, 18, 0], [1, 18, 4]),
 ([3, 19, 0], [1, 19, 4]),
 ([3, 20, 0], [1, 20, 4]),
 ([3, 21, 0], [1, 21, 4]),
 ([3, 22, 0], [1, 22, 4]),
 ([3, 23, 0], [1, 23, 4]),
 ([3, 24, 0], [1, 24, 4]),
 ([3, 25, 0], [1, 25, 4]),
 ([3, 26, 0], [1, 26, 4]),
 ([3, 27, 0], [1, 27, 4]),
 ([3, 28, 0], [1, 28, 4]),
 ([3, 29, 0], [1, 29, 4]),
 ([3, 30, 0], [1, 30, 4]),
 ([3, 31, 0], [1, 31, 4])]

In [19]:

prendus1 = construct_1_particle_interaction(particles['H'],3,1,8).tocoo()
[(base_breakdown(row), base_breakdown(col)) for row, col in list(zip(prendus1.row, prendus1.col))]

[([3, 0, 0], [1, 0, 0]),
 ([3, 1, 0], [1, 1, 0]),
 ([3, 2, 0], [1, 2, 0]),
 ([3, 3, 0], [1, 3, 0]),
 ([3, 4, 0], [1, 4, 0]),
 ([3, 5, 0], [1, 5, 0]),
 ([3, 6, 0], [1, 6, 0]),
 ([3, 7, 0], [1, 7, 0]),
 ([3, 8, 0], [1, 8, 0]),
 ([3, 9, 0], [1, 9, 0]),
 ([3, 10, 0], [1, 10, 0]),
 ([3, 11, 0], [1, 11, 0]),
 ([3, 12, 0], [1, 12, 0]),
 ([3, 13, 0], [1, 13, 0]),
 ([3, 14, 0], [1, 14, 0]),
 ([3, 15, 0], [1, 15, 0]),
 ([3, 16, 0], [1, 16, 0]),
 ([3, 17, 0], [1, 17, 0]),
 ([3, 18, 0], [1, 18, 0]),
 ([3, 19, 0], [1, 19, 0]),
 ([3, 20, 0], [1, 20, 0]),
 ([3, 21, 0], [1, 21, 0]),
 ([3, 22, 0], [1, 22, 0]),
 ([3, 23, 0], [1, 23, 0]),
 ([3, 24, 0], [1, 24, 0]),
 ([3, 25, 0], [1, 25, 0]),
 ([3, 26, 0], [1, 26, 0]),
 ([3, 27, 0], [1, 27, 0]),
 ([3, 28, 0], [1, 28, 0]),
 ([3, 29, 0], [1, 29, 0]),
 ([3, 30, 0], [1, 30, 0]),
 ([3, 31, 0], [1, 31, 0]),
 ([3, 0, 1], [1, 0, 1]),
 ([3, 1, 1], [1, 1, 1]),
 ([3, 2, 1], [1, 2, 1]),
 ([3, 3, 1], [1, 3, 1]),
 ([3, 4, 1], [1, 4, 1]),
 ([3, 5, 1], [1, 5, 1]),
 ([3, 

In [31]:
# base
def base_breakdown(x):
    return [int((x % bases[i+1])/bases[i]) for i in range(particle_types)]

In [92]:
particle1 = particles['H']
particle2 = particles['Li']

b1 = particle1['no_spatial_orbitals'] # number of spatial basis functions for each particle
b2 = particle2['no_spatial_orbitals']

cmb_int = electron_repulsion_integral(particle1['basis'] + particle2['basis'], transform=block_diag(particle1['transform'], particle2['transform']), notation='chemist')[0:b1, 0:b1, b1:b1+b2, b1:b1+b2]
cmb_int *= particle1['properties']['charge'] * particle2['properties']['charge']

print(cmb_int.shape)
print(cmb_int[0,0,0,0])


(8, 8, 8, 8)
0.9953176380940549


In [268]:
particle = particles['e']
t = 14
el_orig = 0
fu = 0

diffs = get_diff_states(particle, el_orig, 0)



print(diffs[fu])
print(particle['states'][el_orig])
print(particle['states'][diffs[fu][0]])

print('\t'.join([str(i) for i in range(t) if particle['states'][el_orig][i] == 1]))
print('\t'.join([str(i) for i in range(t) if particle['states'][diffs[fu][0]][i] == 1]))

(0, [0, 1, 2, 3], (), (), 1)
[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
0	1	2	3
0	1	2	3


In [25]:
particle = particles['H']
print(particle['states'][0])
print(particle['states'][2])

[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [107]:
print(particles['H']['no_spin_orbitals'])

32
